# SampleRepertoire — Overview

Loads multi-locus paired immune repertoires from the SRX tarball and visualises
the per-chain composition of each sample as two stacked bar charts:

* **Top** — total read count (`duplicate_count`) per locus per sample
* **Bottom** — unique clonotype count per locus per sample

Data source: `tests/srx_repertoires/samples.tar.gz`  
Metadata: `tests/srx_repertoires/meta.tsv` (columns: `PMID`, `Run`, `BioProject`, `Sample`)

In [7]:
import sys
sys.path.insert(0, "..")

In [8]:
import io
import tarfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

from mir.common.clonotype import Clonotype
from mir.common.repertoire import SampleRepertoire, infer_locus

ImportError: cannot import name 'is_coding' from 'mir.basic.mirseq' (/Users/mikesh/vcs/mirpy/notebooks/../mir/basic/mirseq.cpython-312-darwin.so)

## Configuration

In [ ]:
TARBALL  = Path("../tests/srx_repertoires/samples.tar.gz")
META_CSV = Path("../tests/srx_repertoires/meta.tsv")

# Loci shown in the plot (ordered and coloured consistently)
LOCI_ORDER  = ["TRA", "TRB", "TRG", "TRD", "IGH", "IGK", "IGL"]
LOCI_COLORS = {
    "TRA": "#4e79a7",
    "TRB": "#f28e2b",
    "TRG": "#59a14f",
    "TRD": "#76b7b2",
    "IGH": "#e15759",
    "IGK": "#b07aa1",
    "IGL": "#ff9da7",
}

# Number of samples to visualise
N_SAMPLES = 30

## Load SampleRepertoires

Each TSV in the tarball follows the AIRR Rearrangement Schema with `v_call` / `j_call` columns.  
Locus is inferred from the gene prefix (`TRBV…` → `TRB`, `TRAV…` → `TRA`, etc.).

In [ ]:
_CALL_RENAMES = {"v_call": "v_gene", "j_call": "j_gene", "c_call": "c_gene"}


def _parse_run(tar: tarfile.TarFile, run_id: str) -> SampleRepertoire:
    """Extract one run TSV from *tar* and return a :class:`SampleRepertoire`."""
    raw = tar.extractfile(f"./{run_id}.tsv").read()
    df  = pd.read_csv(io.BytesIO(raw), sep="\t").rename(columns=_CALL_RENAMES)
    clonotypes = [
        Clonotype(
            duplicate_count=int(row.get("duplicate_count", 1)),
            junction=str(row.get("junction", "")),
            junction_aa=str(row.get("junction_aa", "")),
            v_gene=str(row.get("v_gene", "")),
            j_gene=str(row.get("j_gene", "")),
            c_gene=str(row.get("c_gene", "")),
            locus=infer_locus(str(row.get("v_gene", ""))),
        )
        for _, row in df.iterrows()
    ]
    return SampleRepertoire.from_clonotypes(clonotypes, sample_id=run_id)


def load_samples(
    tarball: Path,
    meta: pd.DataFrame,
    n: int = N_SAMPLES,
) -> list[SampleRepertoire]:
    """Load the first *n* runs present in *tarball* according to *meta*."""
    with tarfile.open(tarball, "r:gz") as tar:
        available = {Path(m.name).stem for m in tar.getmembers() if m.name.endswith(".tsv")}
        run_ids   = [r for r in meta["Run"] if r in available][:n]
        samples   = [_parse_run(tar, rid) for rid in run_ids]
    print(f"Loaded {len(samples)} samples.")
    return samples

In [ ]:
meta    = pd.read_csv(META_CSV, sep="\t")
samples = load_samples(TARBALL, meta, n=N_SAMPLES)

## Build summary table

One row per `(sample_id, locus)` with `duplicate_count` and `clonotype_count`.

In [ ]:
def build_summary(samples: list[SampleRepertoire]) -> pd.DataFrame:
    """Return a long-form DataFrame with per-locus counts for every sample."""
    rows = []
    for sr in samples:
        for locus, lr in sr.loci.items():
            rows.append({
                "sample_id":       sr.sample_id,
                "locus":           locus,
                "duplicate_count": lr.duplicate_count,
                "clonotype_count": lr.clonotype_count,
            })
    return pd.DataFrame(rows)


summary = build_summary(samples)
summary.head()

### Locus coverage

Which loci are absent from at least one sample?

In [ ]:
coverage = (
    summary
    .pivot_table(index="sample_id", columns="locus", values="clonotype_count", fill_value=0)
    .reindex(columns=[l for l in LOCI_ORDER if l in summary["locus"].unique()], fill_value=0)
)

missing_per_sample = (coverage == 0).sum(axis=1)
samples_with_gaps  = missing_per_sample[missing_per_sample > 0]

if not samples_with_gaps.empty:
    print(f"{len(samples_with_gaps)}/{len(coverage)} samples have at least one missing locus:")
    for sid, n_miss in samples_with_gaps.items():
        absent = [l for l in coverage.columns if coverage.loc[sid, l] == 0]
        print(f"  {sid}: missing {absent}")
else:
    print("All loci present in every sample.")

## Stacked bar charts

In [ ]:
def _pivot(summary: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Pivot *summary* into a wide matrix for stacked-bar plotting."""
    present_loci = [l for l in LOCI_ORDER if l in summary["locus"].unique()]
    return (
        summary
        .pivot_table(index="sample_id", columns="locus", values=value_col, fill_value=0)
        .reindex(columns=present_loci, fill_value=0)
    )


def plot_stacked_bars(
    summary: pd.DataFrame,
    *,
    figsize: tuple[int, int] = (14, 8),
) -> None:
    """Draw two stacked bar charts (reads and clonotypes) side by side.

    Parameters
    ----------
    summary:
        Long-form DataFrame with columns ``sample_id``, ``locus``,
        ``duplicate_count``, ``clonotype_count``.
    figsize:
        Overall figure size ``(width, height)`` in inches.
    """
    reads      = _pivot(summary, "duplicate_count")
    clonotypes = _pivot(summary, "clonotype_count")
    colors     = [LOCI_COLORS[l] for l in reads.columns]

    fig, (ax_reads, ax_clones) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    reads.plot(
        kind="bar", stacked=True, ax=ax_reads,
        color=colors, legend=False, width=0.8,
    )
    ax_reads.set_ylabel("Total reads (duplicate_count)")
    ax_reads.set_title("Per-chain read depth across samples")
    ax_reads.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    clonotypes.plot(
        kind="bar", stacked=True, ax=ax_clones,
        color=colors, legend=True, width=0.8,
    )
    ax_clones.set_ylabel("Unique clonotypes")
    ax_clones.set_title("Per-chain clonotype richness across samples")
    ax_clones.set_xlabel("Sample (Run ID)")
    ax_clones.tick_params(axis="x", rotation=45)
    ax_clones.legend(
        title="Locus", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False
    )

    fig.tight_layout()
    plt.show()


plot_stacked_bars(summary)